# AAROH - Phase 1: Content Ingestion & Concept Extraction

This notebook demonstrates the end-to-end Phase 1 pipeline:
1. **Raw Document Ingestion & Parsing**: PyMuPDF extraction with layout and page awareness.
2. **Pedagogical Chunking**: Segmenting curriculum lessons into bite-sized learning chunks.
3. **LLM Concept Extraction**: Extracting atomic concepts, Bloom's levels, difficulty ratings, and rural/tribal real-world context anchors.
4. **Vector Embedding & Storage**: Indexing chunks and concepts into PostgreSQL (`pgvector`).
5. **Curriculum Vector Search**: Semantic retrieval for localization and RAG.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.aaroh.pipeline.content_ingestion_pipeline import ContentIngestionPipeline
from src.aaroh.database.models import DocumentModel, ConceptModel, ContentChunkModel
import pandas as pd

## 1. Run Pipeline on Sample NCERT Lesson

In [ ]:
pipeline = ContentIngestionPipeline()
sample_pdf = Path("../data/raw/sample_class3_evs_water.pdf")

result = pipeline.run_pipeline(
    file_path=sample_pdf,
    grade_hint="Grade 3",
    subject_hint="Environmental Studies"
)

print(f"Document ID: {result.document_id}")
print(f"Status: {result.status}")
print(f"Total Pages: {result.total_pages}")
print(f"Total Chunks: {result.total_chunks}")
print(f"Total Concepts Extracted: {result.total_concepts}")

## 2. Inspect Extracted Concepts & Tribal/Rural Anchors

In [ ]:
session = pipeline.db_manager.get_session()
concepts = session.query(ConceptModel).filter_by(document_id=result.document_id).all()

data = []
for c in concepts:
    data.append({
        "Code": c.concept_code,
        "Concept": c.name,
        "Bloom Level": c.blooms_level,
        "Difficulty (1-5)": c.difficulty_level,
        "Rural/Tribal Anchors": ", ".join(c.rural_tribal_anchors or []),
        "Prerequisites": ", ".join(c.prerequisite_concepts or [])
    })
session.close()

pd.DataFrame(data)

## 3. Perform Vector Similarity Search on Curriculum Chunks

In [ ]:
query = "clay pot cooling and water conservation"
results = pipeline.rag_component.search_similar_content(query=query, top_k=3)

for r in results:
    print(f"Similarity: {r['similarity']} | Page: {r['page_number']}")
    print(f"{r['content']}\n" + "-"*40)